# Notebook B: train and test ResNet50 and EfficientNet-B0 (Kaggle)

Step 2 of the Kaggle pipeline. Trains both models on the slices made by notebook A (`nfti-to-jpeg-conversion.ipynb`), evaluates them on the held-out test patients, compares them, and runs the shortcut checks.

**Set up once:**
1. *Add Input* > **Your Work** > **Notebooks** > notebook A. Its output `data_real/` is found automatically.
2. Settings: **Accelerator: GPU T4** (one T4 is enough), **Internet on** (to fetch the code and the ImageNet weights).

**Run:** **Save Version > Save & Run All (Commit)**. Results land in this version's output under `outputs/`: checkpoints, training curves, confusion matrices, metrics CSVs, the comparison chart and the shortcut-check results.

**After re-running notebook A:** in this notebook's input panel, update notebook A's input to the new version, then commit again.

**Re-running only the checks:** also add a previous version of *this* notebook as an input. Its checkpoints are reused when their classes match the data, so training is skipped.

In [ ]:
# ---------- Setup: GPU, data, code ----------
import glob, os, shutil, subprocess, sys
import torch

MODELS = ["resnet50", "efficientnet_b0"]
HEAD_EPOCHS, EPOCHS, BATCH_SIZE = 5, 10, 32
BRANCH = "main"
REPO_DIR = "/tmp/repo"                       # code stays out of the saved output
OUTPUT_DIR = "/kaggle/working/outputs"       # everything saved with the version


def find_data_dir():
    """The data_real folder from notebook A's output, wherever Kaggle mounted it."""
    hits = [os.path.dirname(d.rstrip("/")) for d in glob.glob("/kaggle/input/**/Training/", recursive=True)
            if os.path.isdir(os.path.join(os.path.dirname(d.rstrip("/")), "Testing"))]
    if not hits:
        raise FileNotFoundError("No data_real (Training/ + Testing/) under /kaggle/input. "
                                "Add notebook A's output with 'Add Input' > Your Work > Notebooks.")
    return sorted(hits, key=len)[0]


def run(cmd):
    """Run a command, stream its output, and stop the notebook if it fails."""
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (turn on GPU T4 in Settings)")
DATA_DIR = find_data_dir()
print("data:", DATA_DIR, "| classes:", sorted(os.listdir(f"{DATA_DIR}/Training")))

run(["rm", "-rf", REPO_DIR])
run(["git", "clone", "--depth", "1", "-b", BRANCH,
     "https://github.com/imfonggg/AI-for-Brain-Tumor-Detection.git", REPO_DIR])
os.makedirs(OUTPUT_DIR, exist_ok=True)
shutil.rmtree(f"{REPO_DIR}/outputs", ignore_errors=True)   # scripts write to outputs/, so point it at OUTPUT_DIR
os.symlink(OUTPUT_DIR, f"{REPO_DIR}/outputs")
os.chdir(REPO_DIR)

# Reuse checkpoints from a previous version of this notebook, if one is attached as input
for ckpt in glob.glob("/kaggle/input/**/outputs/*_best.pt", recursive=True):
    name = os.path.basename(ckpt)
    if not os.path.exists(f"{OUTPUT_DIR}/{name}"):
        shutil.copy(ckpt, OUTPUT_DIR)
        curves = ckpt.replace("_best.pt", "_curves.png")
        if os.path.exists(curves):
            shutil.copy(curves, OUTPUT_DIR)
        print("reusing", ckpt)

In [ ]:
# ---------- Train each model, then evaluate it on the held-out Testing/ patients ----------
# A reused checkpoint is kept only if it was trained on the same classes as the current data.
import os, torch
os.chdir(REPO_DIR)
data_classes = sorted(os.listdir(f"{DATA_DIR}/Training"))


def finished_on_same_classes(model):
    if not (os.path.exists(f"outputs/{model}_curves.png") and os.path.exists(f"outputs/{model}_best.pt")):
        return False
    ckpt_classes = torch.load(f"outputs/{model}_best.pt", map_location="cpu").get("class_names")
    if ckpt_classes != data_classes:
        print(f"{model}: saved run used classes {ckpt_classes}, data has {data_classes}; retraining")
        return False
    return True


for model in MODELS:
    if finished_on_same_classes(model):
        print(f"{model}: already trained on {data_classes}, skipping training")
    else:
        run([sys.executable, "-u", "train.py", "--model", model, "--data_dir", DATA_DIR,
             "--head_epochs", str(HEAD_EPOCHS), "--epochs", str(EPOCHS), "--batch_size", str(BATCH_SIZE)])
    run([sys.executable, "-u", "evaluate.py", "--checkpoint", f"outputs/{model}_best.pt",
         "--data_dir", DATA_DIR, "--output_csv", f"outputs/{model}_metrics.csv"])

In [ ]:
# ---------- Side-by-side comparison ----------
os.chdir(REPO_DIR)
run([sys.executable, "compare_models.py", *[f"outputs/{m}_metrics.csv" for m in MODELS]])

from IPython.display import Image, display
display(Image("outputs/model_comparison.png"))
for model in MODELS:
    display(Image(f"outputs/{model}_curves.png"))
    display(Image(f"outputs/{model}_confusion_matrix.png"))

## Shortcut checks: is the model reading the tumor?

Classes come from different datasets (BraTS, Figshare, IXI), so the models could score well by recognising the *dataset* (brightness, skull or no skull, borders) instead of the tumor. Notebook A now harmonizes every source and gives glioma and meningioma two sources; these checks show whether that worked. A perfect score on one class is a warning sign, not proof of success. These cells test for that, using the trained models in `outputs/`. Run the setup cell first, then any of the three checks. Each check saves its figure or table to `outputs/` in this version's output.

1. **Grad-CAM:** heatmaps of where each model looks, 8 test slices per class. Good: the heat sits on the tumor (or spread over brain tissue for no tumor). Bad: the heat sits on the skull, the image edge or the black background.
2. **Source test:** a simple classifier that only sees plain image statistics (brightness histogram, contrast, how much black border) tries to guess which dataset each slice came from (from `manifest.csv`), and lists classes that still have a single source. If it gets close to 100%, a shortcut is *available* to the CNNs.
3. **Masked test:** re-runs each model on altered test slices. If a class is still recognised after the brain center is blanked out, after the pixels are shuffled, or after the model's own hot spot is erased, the model is not relying on the tumor for that class.

In [ ]:
# ---------- Shortcut checks: setup (run once before the three checks) ----------
import glob, os, random, re, sys
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

REPO_DIR = "/tmp/repo"
OUT = "outputs"
MODELS = ["resnet50", "efficientnet_b0"]
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
from data_loader import get_transforms
from model import build_model

def find_data_dir():
    """The data_real folder from notebook A's output, wherever Kaggle mounted it."""
    hits = [os.path.dirname(d.rstrip("/")) for d in glob.glob("/kaggle/input/**/Training/", recursive=True)
            if os.path.isdir(os.path.join(os.path.dirname(d.rstrip("/")), "Testing"))]
    if not hits:
        raise FileNotFoundError("No data_real (Training/ + Testing/) under /kaggle/input. "
                                "Add notebook A's output with 'Add Input' > Your Work > Notebooks.")
    return sorted(hits, key=len)[0]


DATA_DIR = find_data_dir()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_tf = get_transforms(train=False)
CLASSES = sorted(os.listdir(f"{DATA_DIR}/Testing"))


def load_model(name):
    ckpt = torch.load(f"{OUT}/{name}_best.pt", map_location=device)
    model = build_model(name, num_classes=len(ckpt["class_names"]), pretrained=False)
    model.load_state_dict(ckpt["model_state_dict"])
    return model.to(device).eval(), ckpt["class_names"]


def target_layer(model, name):
    """Last convolutional block, where Grad-CAM is usually taken."""
    return model.layer4[-1] if name.startswith("resnet") else model.features[-1]


def sample_test_files(per_class, seed=0):
    rng = random.Random(seed)
    files = {}
    for cls in CLASSES:
        paths = sorted(glob.glob(f"{DATA_DIR}/Testing/{cls}/*.jpg"))
        files[cls] = rng.sample(paths, min(per_class, len(paths)))
    return files


def load_gray(path):
    """Grayscale uint8 array at the model's input size."""
    return np.asarray(Image.open(path).convert("L").resize((224, 224)), dtype=np.uint8)


def to_batch(gray_images):
    return torch.stack([eval_tf(Image.fromarray(g).convert("RGB")) for g in gray_images]).to(device)


@torch.no_grad()
def predict(model, gray_images, batch_size=32):
    preds = []
    for i in range(0, len(gray_images), batch_size):
        preds.append(model(to_batch(gray_images[i:i + batch_size])).argmax(1).cpu())
    return torch.cat(preds).numpy()


def gradcam(model, name, gray):
    """Grad-CAM heatmap (224x224, 0..1) for the predicted class, plus that class index and confidence."""
    store = {}

    def hook(_module, _inputs, output):
        store["act"] = output
        output.register_hook(lambda g: store.setdefault("grad", g))

    handle = target_layer(model, name).register_forward_hook(hook)
    model.zero_grad(set_to_none=True)
    logits = model(to_batch([gray]))
    idx = int(logits.argmax(1))
    logits[0, idx].backward()
    handle.remove()
    weights = store["grad"].mean(dim=(2, 3), keepdim=True)
    cam = (weights * store["act"]).sum(1, keepdim=True).relu()
    cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[0, 0]
    cam = cam.detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, idx, torch.softmax(logits, 1)[0, idx].item()


missing = [m for m in MODELS if not os.path.exists(f"{OUT}/{m}_best.pt")]
if missing:
    raise FileNotFoundError(f"No trained checkpoint for {missing} in {OUT}/. Run the training cell first.")
print("classes:", CLASSES, "| test slices:", {c: len(os.listdir(f"{DATA_DIR}/Testing/{c}")) for c in CLASSES})

In [ ]:
# ---------- Check 1: Grad-CAM, 8 test slices per class, both models ----------
if "load_model" not in globals():
    raise RuntimeError("Run the shortcut-checks setup cell first.")

PER_CLASS = 8
files = sample_test_files(PER_CLASS, seed=1)
for name in MODELS:
    model, class_names = load_model(name)
    fig, axes = plt.subplots(len(CLASSES), PER_CLASS, figsize=(2.2 * PER_CLASS, 2.5 * len(CLASSES)),
                             squeeze=False)
    background_heat = {}
    for row, cls in enumerate(CLASSES):
        shares = []
        for col, path in enumerate(files[cls]):
            gray = load_gray(path)
            cam, idx, conf = gradcam(model, name, gray)
            # Share of the heat that falls on near-black pixels (background / outside the brain)
            shares.append(cam[gray < 10].sum() / (cam.sum() + 1e-8))
            ax = axes[row, col]
            ax.imshow(gray, cmap="gray")
            ax.imshow(cam, cmap="jet", alpha=0.4)
            ok = "" if class_names[idx] == cls else "  (WRONG)"
            ax.set_title(f"true {cls}\npred {class_names[idx]} {conf:.2f}{ok}", fontsize=7)
        for ax in axes[row]:
            ax.axis("off")
        background_heat[cls] = float(np.mean(shares)) if shares else float("nan")
    fig.suptitle(f"Grad-CAM: {name}", fontsize=12)
    fig.tight_layout()
    path = f"{OUT}/gradcam_{name}.png"
    fig.savefig(path, dpi=110)
    plt.show()
    print(f"{name}: saved {path}")
    print("  share of heat on black background (lower is better):",
          {c: f"{v:.0%}" for c, v in background_heat.items()})

In [ ]:
# ---------- Check 2: can plain image statistics tell the datasets apart? ----------
# Uses data_real/manifest.csv for each slice's source. Lists classes that still come from one source.
import subprocess, sys
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "shortcut_checks.py", "source", "--data_dir", DATA_DIR,
                "--output_dir", OUT], check=False)

In [ ]:
# ---------- Check 3: does each model still recognise a class with the tumor area removed? ----------
if "load_model" not in globals():
    raise RuntimeError("Run the shortcut-checks setup cell first.")
import pandas as pd

PER_CLASS = 50
files = sample_test_files(PER_CLASS, seed=2)
yy, xx = np.mgrid[:224, :224]
CENTER = ((yy - 112) / 80.0) ** 2 + ((xx - 112) / 80.0) ** 2 <= 1   # central ellipse, ~40% of the image


def fill_value(gray):
    brain = gray[gray > 10]
    return int(np.median(brain)) if brain.size else 0


def blank_center(gray):
    """Keep only the outer ring (edges, skull, background)."""
    out = gray.copy()
    out[CENTER] = 0
    return out


def shuffle_pixels(gray, rng):
    """Same brightness histogram, no shapes at all."""
    return rng.permutation(gray.ravel()).reshape(gray.shape)


def blank_hotspot(gray, cam, top=0.10):
    """Erase the 10% of pixels the model looks at most, filling with typical brain intensity."""
    out = gray.copy()
    out[cam >= np.quantile(cam, 1 - top)] = fill_value(gray)
    return out


rows = []
for name in MODELS:
    model, class_names = load_model(name)
    rng = np.random.default_rng(0)
    for cls in CLASSES:
        if cls not in class_names:
            continue
        target = class_names.index(cls)
        grays = [load_gray(p) for p in files[cls]]
        variants = {
            "original": grays,
            "center blanked": [blank_center(g) for g in grays],
            "pixels shuffled": [shuffle_pixels(g, rng) for g in grays],
            "hot spot erased": [blank_hotspot(g, gradcam(model, name, g)[0]) for g in grays],
        }
        row = {"model": name, "class": cls, "n": len(grays)}
        for label, imgs in variants.items():
            row[label] = float((predict(model, imgs) == target).mean())
        rows.append(row)

table = pd.DataFrame(rows)
table.to_csv(f"{OUT}/masked_test.csv", index=False)
pd.options.display.float_format = "{:.0%}".format
display(table)
print("Each number is how often the model still predicts the TRUE class after the change.")
print("For a model that reads the tumor: 'original' high, and the other three columns clearly lower")
print("for the tumor classes. A tumor class that stays high with the center blanked or pixels")
print("shuffled is being recognised from edges, skull or brightness instead of the tumor.")
fig, ax = plt.subplots(figsize=(10, 3.5))
table.set_index(["model", "class"])[["original", "center blanked", "pixels shuffled", "hot spot erased"]] \
    .plot.bar(ax=ax, ylim=(0, 1), rot=30)
ax.set_ylabel("still predicts true class")
ax.set_title("Masked test")
fig.tight_layout()
fig.savefig(f"{OUT}/masked_test.png", dpi=110)
plt.show()

## Pass/fail: pixel-shuffle test

Shuffling pixels destroys all anatomy and keeps only brightness, so a model that reads the disease should predict each class about 1/K of the time (25% with 4 classes). **PASS** needs every class within 15 points of chance. The shuffled confusion matrix shows whether a failure is a fingerprint (strong diagonal, balanced accuracy above chance) or just a default answer for noise (one full column). Results: `outputs/shuffle_test_<model>.csv/.json`.

In [ ]:
# ---------- Pass/fail: pixel-shuffle test for both models ----------
import subprocess, sys
os.chdir(REPO_DIR)
verdicts = {}
for name in MODELS:
    r = subprocess.run([sys.executable, "shortcut_checks.py", "shuffle", "--checkpoint", f"{OUT}/{name}_best.pt",
                        "--data_dir", DATA_DIR, "--output_dir", OUT, "--strict"])
    verdicts[name] = "PASS" if r.returncode == 0 else "FAIL"
print("\nShuffle test:", verdicts)

## Leave-one-source-out (optional, GPU heavy)

Trains once per source with that source left out, then tests on it. The source fingerprint is useless on an unseen source, so this accuracy is the honest one; expect it to be lower than the normal test accuracy. Sources whose classes appear in no other source are skipped and listed. Set `RUN_LOSO = True` to run it (roughly one short training run per source; 7 sources take about an hour per model on a T4).

In [ ]:
# ---------- Leave-one-source-out ----------
RUN_LOSO = False
LOSO_MODEL, LOSO_HEAD_EPOCHS, LOSO_EPOCHS = "resnet50", 3, 5
import subprocess, sys
os.chdir(REPO_DIR)
plan = [sys.executable, "-u", "loso.py", "--data_dir", DATA_DIR, "--model", LOSO_MODEL,
        "--head_epochs", str(LOSO_HEAD_EPOCHS), "--epochs", str(LOSO_EPOCHS)]
subprocess.run(plan + ["--dry_run"], check=True)
if RUN_LOSO:
    subprocess.run(plan, check=True)
    import pandas as pd
    display(pd.read_csv(f"{OUT}/loso_{LOSO_MODEL}_summary.csv"))